# Delta — Jupyter Launcher

Launcher interactivo del motor **Delta** (señales SNT para cripto y bolsa),
clonado del Jupyter launcher de Sentinel Omega y adaptado a este módulo.

Corre las celdas en orden:
1. **Setup & Imports** — configura el entorno
2. **Inicializar motor** — constantes de fricción y b esperada por mercado
3. **Cargar datos** — precios reales (CoinGecko + Yahoo, sin API key) o demo sintético
4. **Analizar un par** — pipeline completo sobre un hub/shadow
5. **Barrido completo** — todas las señales (cripto + bolsa US + bolsa MX)
6. **Consultar resultados** — DataFrame + JSON de señales
7. **Visualizar** — b por par vs fricción esperada, rolling-b
8. **Guardar & cierre**


## 1. Setup & Imports


In [ ]:
import sys
from pathlib import Path

# El notebook vive en delta/notebooks/ — agrega delta/ al path
DELTA_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd() / 'delta'
sys.path.insert(0, str(DELTA_DIR))

import numpy as np
print(f'Delta dir: {DELTA_DIR}')
print(f'numpy {np.__version__}')


In [ ]:
# Módulos del motor Delta (SNT portado, autocontenido — sin dependencia cross-repo)
from snt_market_core import fit_satellization, classify_regime, rolling_b
from market_mapping import build_dominance, MARKET_FRICTION
from delta_engine import analyze_pair, DeltaSignal
print('Motor Delta cargado')


## 2. Inicializar motor — fricción y b esperada

El hallazgo central de SNT (ρ = −0.68): la fricción institucional predice el
exponente de satelización. Cripto = fricción BAJA (b esperada ≈ 0.60);
bolsa = fricción MEDIA (b esperada ≈ 0.30). La desviación grande contra esa
b-nula es la **anomalía** operable.


In [ ]:
for market, info in MARKET_FRICTION.items():
    print(f'{market}: {info}')


## 3. Cargar datos — reales (sin API key) o demo

`USE_REAL_DATA = False` usa series sintéticas (sin red, igual que `demo_delta.py`).
`True` baja cierres diarios reales: CoinGecko (cripto) + Yahoo Finance (bolsa).


In [ ]:
USE_REAL_DATA = False   # cambia a True para bajar precios reales

if USE_REAL_DATA:
    from data_adapters import fetch_crypto, fetch_yahoo, align_pair
    btc = fetch_crypto('bitcoin', days=180)
    eth = fetch_crypto('ethereum', days=180)
    hub_c, shadow_c = align_pair(btc, eth)
    spx = fetch_yahoo('^GSPC', rng='6mo')
    aapl = fetch_yahoo('AAPL', rng='6mo')
    hub_b, shadow_b = align_pair(spx, aapl)
    print(f'BTC/ETH: {len(hub_c)} días · SPX/AAPL: {len(hub_b)} días')
else:
    rng = np.random.default_rng(42)
    t = np.arange(1, 181, dtype=float)
    hub_c = 100 * t**0.35 * np.exp(rng.normal(0, 0.02, t.size))
    shadow_c = 80 * t**0.05 * np.exp(rng.normal(0, 0.03, t.size))
    hub_b = 400 * t**0.12 * np.exp(rng.normal(0, 0.01, t.size))
    shadow_b = 150 * t**0.20 * np.exp(rng.normal(0, 0.02, t.size))
    print('Series sintéticas listas (180 días)')


## 4. Analizar un par — pipeline completo → DeltaSignal


In [ ]:
sig = analyze_pair(hub_c, shadow_c, hub='BTC', shadow='ETH', market='crypto')
print(f'b = {sig.b:+.3f}  (esperada {sig.expected_b:+.2f})')
print(f'régimen: {sig.regime} · R²={sig.r_squared:.3f} · p={sig.p_value:.4f}')
print(f'anomalía: {sig.anomaly_score:.2f} · leapfrog: {sig.leapfrog}')
print(f'dirección: {sig.direction} · confianza: {sig.confidence:.2f}')


## 5. Barrido completo — cripto + bolsa (reales)

Equivale a `python run_real_delta.py`: BTC vs top alts, S&P 500 vs caps US,
IPC vs emisoras BMV. **Requiere red** (tarda unos minutos por el rate-limit
de CoinGecko). Deja `RUN_FULL_SWEEP = False` si solo estás explorando.


In [ ]:
RUN_FULL_SWEEP = False

if RUN_FULL_SWEEP:
    import run_real_delta
    run_real_delta.main()   # escribe real_delta_signals.json (~23 señales)
else:
    print('Sweep omitido — usa el JSON existente en la siguiente celda')


## 6. Consultar resultados


In [ ]:
import json, pandas as pd

signals_path = DELTA_DIR / 'real_delta_signals.json'
if signals_path.exists():
    data = json.loads(signals_path.read_text())
    df = pd.DataFrame(data['signals'] if isinstance(data, dict) and 'signals' in data else data)
    display(df.sort_values('anomaly_score', ascending=False).head(15))
else:
    print('Sin JSON aún — corre el sweep (celda 5) o analyze_pair (celda 4)')
    df = None


## 7. Visualizar — b por par vs b esperada por fricción


In [ ]:
import matplotlib.pyplot as plt

if df is not None and len(df):
    d = df.sort_values('b')
    labels = d['shadow'] + ' (' + d['market'] + ')'
    colors = ['#C0392B' if lf else '#2C7FB8' for lf in d.get('leapfrog', [False]*len(d))]
    fig, ax = plt.subplots(figsize=(9, max(4, len(d)*0.35)))
    ax.barh(labels, d['b'], color=colors)
    for exp_b, mk in [(0.60, 'crypto'), (0.30, 'bolsa')]:
        ax.axvline(exp_b, ls='--', lw=1, alpha=0.6, label=f'b esperada {mk} ({exp_b})')
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('exponente b  (rojo = leapfrog)')
    ax.set_title('Delta — satelización observada vs fricción esperada')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    # rolling-b del par analizado en la celda 4
    dom = build_dominance(hub_c, shadow_c, 'BTC', 'ETH', 'crypto')
    bs = rolling_b(dom.ratio, window=30)
    plt.figure(figsize=(9,4))
    plt.plot(bs, lw=1.5)
    plt.axhline(0, color='k', lw=0.8)
    plt.title('Rolling-b (ventana 30) — cambios de régimen')
    plt.xlabel('día'); plt.ylabel('b'); plt.tight_layout(); plt.show()


## 8. Guardar & cierre


In [ ]:
# Las señales quedan en real_delta_signals.json (sin precios crudos — solo la esencia).
# Principio del proyecto: aprender la esencia, soltar el bulto.
print('Sesión Delta terminada.')
